# DORA RAG — Phase 4 (v2): Retrieval Pipeline

**Fix from v1:** `QueryFusionRetriever` internally calls `Settings.llm` even
when `num_queries=1`, causing LlamaIndex to fall back to OpenAI and fail.
This version explicitly sets `Settings.llm` to a mock/Groq before any
retriever is constructed, preventing the OpenAI lookup entirely.

**Prerequisites:** Phases 1, 2, 3 complete. `dora_rag/index_config.json` must exist.

In [1]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "llama-index",
    "llama-index-llms-groq",
    "llama-index-embeddings-huggingface",
    "llama-index-vector-stores-chroma",
    "llama-index-retrievers-bm25",
    "chromadb",
    "sentence-transformers",
    "torch",
    "rank-bm25",
    "PyStemmer",
])
print("Dependencies installed.")


Dependencies installed.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [16]:
import os
# Cell 2 — Imports, config, LLM setup (Groq only)
#
# LLM strategy — both on Groq free tier:
#   Settings.llm  → Groq 8b-instant   (router internals — lightweight)
#   llm           → Groq 70b-versatile (HyDE hypothesis generation)
#
# Why two models:
#   Settings.llm must be set before any LlamaIndex object is built to prevent
#   the OpenAI fallback. We use the lightweight 8b model here because
#   QueryFusionRetriever only calls it for internal bookkeeping (num_queries=1
#   means no actual LLM generation happens during retrieval).
#   The 70b model is used for HyDE hypothesis generation where quality matters.

import json
import time
import re as _re
import datetime
import torch
from pathlib import Path
from collections import Counter

from llama_index.core import Settings
from llama_index.llms.groq import Groq

GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')  # set in your environment or a .env file

# Router LLM: lightweight 8b — prevents OpenAI fallback, never used for generation
router_llm = Groq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)
Settings.llm = router_llm
print("✓ Settings.llm set to Groq 8b — OpenAI fallback disabled")

# Generation LLM: 70b — used for HyDE hypothesis generation
llm = Groq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
print("✓ Generation LLM: Groq llama-3.3-70b-versatile")

# ── Paths and config ──────────────────────────────────────────────────────────
BASE_DIR    = Path("./dora_rag")
CHUNKS_DIR  = BASE_DIR / "dora_chunks"
CHUNKS_FILE = CHUNKS_DIR / "dora_chunks.json"
CONFIG_FILE = BASE_DIR / "index_config.json"

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"index_config.json not found at {CONFIG_FILE}\n"
        "Please run 03_indexing_v2.ipynb first."
    )

config = json.loads(CONFIG_FILE.read_text())
print(f"\nIndex config loaded:")
print(f"  Model      : {config['embed_model']}")
print(f"  Collection : {config['chroma_collection']}")
print(f"  Vectors    : {config['total_vectors']}")
print(f"  Levels     : {config.get('corpus_levels', {})}")
print(f"  Doc types  : {config['doc_types_indexed']}")

CHROMA_DIR        = Path(config["chroma_dir"])
CHROMA_COLLECTION = config["chroma_collection"]
EMBED_MODEL_NAME  = config["embed_model"]
device            = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device     : {device}")


✓ Settings.llm set to Groq 8b — OpenAI fallback disabled
✓ Generation LLM: Groq llama-3.3-70b-versatile

Index config loaded:
  Model      : BAAI/bge-large-en-v1.5
  Collection : dora_regulation
  Vectors    : 381
  Levels     : {'1': 130, '2': 241, '3': 10}
  Doc types  : {'regulation': 114, 'directive': 16, 'RTS': 133, 'ITS': 94, 'delegated_regulation': 14, 'guideline': 7, 'statement': 3}
  Device     : cpu


In [17]:
# Cell 3 — Load embedding model
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print(f"Loading {EMBED_MODEL_NAME} on {device}...")
t0 = time.time()

embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL_NAME,
    query_instruction="Represent this sentence for searching relevant passages: ",
    embed_batch_size=16,
    device=device,
)
Settings.embed_model = embed_model

print(f"✓ Model loaded in {time.time()-t0:.1f}s")

Loading BAAI/bge-large-en-v1.5 on cpu...
✓ Model loaded in 2.1s


In [18]:
# Cell 4 — Load Chroma index
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

chroma_client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
chroma_collection = chroma_client.get_collection(CHROMA_COLLECTION)
vector_store      = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context   = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)

print(f"✓ Index loaded — {chroma_collection.count()} vectors")

✓ Index loaded — 381 vectors


In [19]:
# Cell 5 — Load chunks and build TextNodes for BM25
from llama_index.core.schema import TextNode

if not CHUNKS_FILE.exists():
    raise FileNotFoundError(f"Chunks file not found: {CHUNKS_FILE}")

raw_chunks = json.loads(CHUNKS_FILE.read_text(encoding="utf-8"))

all_nodes = [
    TextNode(
        id_  = c["chunk_id"],
        text = c["text_with_header"],
        metadata = {
            "source_doc":     c["source_doc"],
            "doc_type":       c["doc_type"],
            "level":          c.get("level", 0),
            "article_num":    c["article_num"] if c["article_num"] is not None else -1,
            "article_title":  c["article_title"],
            "paragraph_idx":  c["paragraph_idx"],
            "topic_tags":     "|".join(c["topic_tags"]),
            "entity_scope":   "|".join(c["entity_scope"]),
            "token_estimate": c["token_estimate"],
        }
    )
    for c in raw_chunks
]

print(f"✓ Loaded {len(all_nodes)} nodes")

# Show breakdown
by_source = Counter(c["source_doc"] for c in raw_chunks)
print(f"\nChunks per document:")
for src, count in sorted(by_source.items()):
    print(f"  {src[:50]:<50} {count:>5}")

✓ Loaded 390 nodes

Chunks per document:
  L1_DORA_Directive_2022_2556                           16
  L1_DORA_Regulation_2022_2554                         116
  L2_DR_CTPP_Designation_Criteria                        7
  L2_DR_DORA_Oversight_Fees                              7
  L2_ITS_ICT_Incident_Reporting                         47
  L2_ITS_Register_of_Information                        47
  L2_RTS_ICT_Incident_Classification                    14
  L2_RTS_ICT_Incident_Reporting_Process                  8
  L2_RTS_ICT_Risk_Management                            63
  L2_RTS_ICT_Third_Party_Policy                          8
  L2_RTS_Joint_Examination_Team                          7
  L2_RTS_Oversight_Conditions                           12
  L2_RTS_Subcontracting                                 10
  L2_RTS_TLPT                                           18
  L3_ESAs_Public_Statement_DORA                          3
  L3_Guidelines_Costs_and_Losses                         7


In [21]:
# Cell 6 — Build BM25 retriever
from llama_index.retrievers.bm25 import BM25Retriever

try:
    import Stemmer
    stemmer = Stemmer.Stemmer("english")
    print("Using PyStemmer for BM25")
except ImportError:
    stemmer = None
    print("PyStemmer not found — BM25 will run without stemming (still works)")

bm25_retriever = BM25Retriever.from_defaults(
    nodes=all_nodes,
    similarity_top_k=5,
    stemmer=stemmer,
    language="english",
)

print(f"✓ BM25 retriever built over {len(all_nodes)} nodes")

Using PyStemmer for BM25
✓ BM25 retriever built over 390 nodes


In [22]:
# Cell 7 — Build semantic retriever
from llama_index.core.retrievers import VectorIndexRetriever

semantic_retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5,
)

print("✓ Semantic retriever ready")

✓ Semantic retriever ready


In [23]:
# Cell 8 — Build hybrid retriever
#
# Now safe because Settings.llm was set to Groq in Cell 2.
# QueryFusionRetriever will use Groq if it needs an LLM internally,
# but with num_queries=1 it performs pure RRF fusion without any LLM calls.

from llama_index.core.retrievers import QueryFusionRetriever

hybrid_retriever = QueryFusionRetriever(
    retrievers=[semantic_retriever, bm25_retriever],
    llm=Settings.llm,            # explicitly pass Groq — no ambiguity
    similarity_top_k=5,
    num_queries=1,               # no query expansion — pure RRF fusion
    mode="reciprocal_rerank",
    use_async=False,
    verbose=False,
)

print("✓ Hybrid retriever ready (semantic + BM25, RRF fusion, no OpenAI)")

✓ Hybrid retriever ready (semantic + BM25, RRF fusion, no OpenAI)


In [24]:
# Cell 9 — Display helper
def show_results(results, label: str = ""):
    if label:
        print(f"\n{'═'*65}")
        print(f"  {label}")
        print(f"{'═'*65}")
    print(f"  Results : {len(results)}")
    print(f"  {'#':<3} {'L':>2} {'Type':<22} {'Art':>4}  {'Score':>6}  Source / Title")
    print(f"  {'─'*62}")
    for i, node in enumerate(results, 1):
        m     = node.metadata
        lv    = m.get('level', '?')
        dtype = m.get('doc_type', '?')
        art   = m.get('article_num', '?')
        title = m.get('article_title', '')[:28]
        src   = m.get('source_doc', '')[:28]
        score = getattr(node, 'score', 0.0) or 0.0
        print(f"  {i:<3} {lv:>2} {dtype:<22} {str(art):>4}  {score:>6.4f}  {src} — {title}")
    print()

print("Display helper ready.")

Display helper ready.


In [25]:
# Cell 10 — Compare: semantic vs BM25 vs hybrid on same query
TEST_QUERY = "What are the reporting timelines for major ICT incidents?"
print(f"Query: {TEST_QUERY}\n")

sem_results    = semantic_retriever.retrieve(TEST_QUERY)
bm25_results   = bm25_retriever.retrieve(TEST_QUERY)
hybrid_results = hybrid_retriever.retrieve(TEST_QUERY)

show_results(sem_results,    "SEMANTIC RETRIEVER")
show_results(bm25_results,   "BM25 RETRIEVER")
show_results(hybrid_results, "HYBRID RETRIEVER (recommended)")

sem_ids  = {n.id_ for n in sem_results}
bm25_ids = {n.id_ for n in bm25_results}
print(f"Overlap (semantic ∩ BM25) : {len(sem_ids & bm25_ids)} chunk(s)")
print(f"Unique to semantic        : {len(sem_ids - bm25_ids)}")
print(f"Unique to BM25            : {len(bm25_ids - sem_ids)}")

Query: What are the reporting timelines for major ICT incidents?


═════════════════════════════════════════════════════════════════
  SEMANTIC RETRIEVER
═════════════════════════════════════════════════════════════════
  Results : 5
  #    L Type                    Art   Score  Source / Title
  ──────────────────────────────────────────────────────────────
  1    1 regulation               21  0.7713  L1_DORA_Regulation_2022_2554 — Centralisation of reporting 
  2    2 ITS                       5  0.7673  L2_ITS_ICT_Incident_Reportin — Reclassification of major IC
  3    2 ITS                       9  0.7586  L2_ITS_ICT_Incident_Reportin — Entry into force
  4    2 RTS                      21  0.7560  L2_RTS_ICT_Risk_Management — Access control
  5    2 ITS                       3  0.7559  L2_ITS_ICT_Incident_Reportin — Recurring ICT-related incide


═════════════════════════════════════════════════════════════════
  BM25 RETRIEVER
═════════════════════════════════════════════════════

In [26]:
# Cell 11 — Test across all DORA query types
TEST_QUERIES = [
    ("What must be included in the initial notification of a major ICT incident?", "factual_lookup"),
    ("TLPT requirements for significant institutions",                             "exact_term"),
    ("Article 28 general principles ICT third-party risk management",              "article_reference"),
    ("Does DORA apply to crypto-asset service providers?",                         "applicability"),
    ("Is the appointment of an ICT third-party risk function mandatory?",           "obligation_type"),
    ("What do the technical standards say about subcontracting ICT services?",      "cross_level"),
    ("What does DORA say about GDPR fines?",                                       "negative_test"),
]

for query, qtype in TEST_QUERIES:
    results = hybrid_retriever.retrieve(query)
    print(f"[{qtype.upper()}]")
    print(f"Query: {query}")
    show_results(results)

[FACTUAL_LOOKUP]
Query: What must be included in the initial notification of a major ICT incident?
  Results : 5
  #    L Type                    Art   Score  Source / Title
  ──────────────────────────────────────────────────────────────
  1    1 regulation               19  0.0323  L1_DORA_Regulation_2022_2554 — Reporting of major ICT-relat
  2    2 RTS                       5  0.0167  L2_RTS_ICT_Incident_Reportin — Time limits for the initial 
  3    2 ITS                       9  0.0164  L2_ITS_ICT_Incident_Reportin — Entry into force
  4    2 RTS                       2  0.0164  L2_RTS_ICT_Incident_Reportin — Specific information to be p
  5    2 ITS                       9  0.0161  L2_ITS_ICT_Incident_Reportin — Entry into force

[EXACT_TERM]
Query: TLPT requirements for significant institutions
  Results : 5
  #    L Type                    Art   Score  Source / Title
  ──────────────────────────────────────────────────────────────
  1    1 regulation               27  0.0331  L

In [27]:
# Cell 12 — Filtered retriever factory
from llama_index.core.vector_stores import (
    MetadataFilter,
    MetadataFilters,
    FilterOperator,
)

def make_filtered_retriever(filters: MetadataFilters, top_k: int = 5):
    return VectorIndexRetriever(
        index=index,
        similarity_top_k=top_k,
        filters=filters,
    )

retriever_level1 = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="level", value=1, operator=FilterOperator.EQ)
]))
retriever_level2 = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="level", value=2, operator=FilterOperator.EQ)
]))
retriever_level3 = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="level", value=3, operator=FilterOperator.EQ)
]))
retriever_rts = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="doc_type", value="RTS", operator=FilterOperator.EQ)
]))
retriever_its = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="doc_type", value="ITS", operator=FilterOperator.EQ)
]))

print("✓ Filtered retrievers ready:")
print("  retriever_level1 — Level 1 (regulation + directive)")
print("  retriever_level2 — Level 2 (RTS + ITS)")
print("  retriever_level3 — Level 3 (guidelines + statements)")
print("  retriever_rts    — RTS documents only")
print("  retriever_its    — ITS documents only")

✓ Filtered retrievers ready:
  retriever_level1 — Level 1 (regulation + directive)
  retriever_level2 — Level 2 (RTS + ITS)
  retriever_level3 — Level 3 (guidelines + statements)
  retriever_rts    — RTS documents only
  retriever_its    — ITS documents only


In [28]:
# Cell 13 — Query router (make_retriever pattern)
#
# Uses make_retriever() to build a fresh retriever per query with the correct top_k.
# LlamaIndex retrievers do not reliably update top_k after construction,
# so we build a new one each time rather than patching an existing retriever.

from llama_index.core.vector_stores import (
    MetadataFilter, MetadataFilters, FilterOperator
)
from llama_index.core.retrievers import VectorIndexRetriever, QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever

def make_retriever(level=None, doc_type=None, top_k=8):
    """Build a fresh retriever with the specified top_k and optional metadata filters."""
    filters = []
    if level    is not None: filters.append(MetadataFilter(key="level",    value=level,    operator=FilterOperator.EQ))
    if doc_type is not None: filters.append(MetadataFilter(key="doc_type", value=doc_type, operator=FilterOperator.EQ))
    if filters:
        return VectorIndexRetriever(index=index, similarity_top_k=top_k,
                                    filters=MetadataFilters(filters=filters))
    sem = VectorIndexRetriever(index=index, similarity_top_k=top_k)
    bm  = BM25Retriever.from_defaults(
        nodes=all_nodes, similarity_top_k=top_k,
        stemmer=stemmer, language="english"
    )
    return QueryFusionRetriever(
        retrievers=[sem, bm], llm=Settings.llm,
        similarity_top_k=top_k, num_queries=1,
        mode="reciprocal_rerank", use_async=False, verbose=False,
    )


def route_query(query: str):
    """Route query to appropriate retriever. Returns (retriever, scope)."""
    q = query.lower()
    if any(k in q for k in ["regulatory technical standard", " rts ", "rts on",
                             "implementing technical standard", " its ", "its on",
                             "technical standard", "delegated regulation"]):
        return make_retriever(level=2, top_k=12), "Level 2 (RTS/ITS)"
    # TLPT: full corpus so Article 26 from L1 is also retrieved
    if any(k in q for k in ["tlpt", "penetration test", "threat-led", "tiber"]):
        return make_retriever(top_k=12), "Full corpus (TLPT)"
    if any(k in q for k in ["subcontract", "chain", "group entit",
                             "concentrat", "third-party provider"]):
        return make_retriever(top_k=12), "Full corpus (adversarial)"
    if any(k in q for k in ["guideline", "aggregate", "oversight cooperation",
                             "public statement"]):
        return make_retriever(level=3, top_k=8), "Level 3 (guidelines)"
    if any(k in q for k in ["template", "reporting form"]):
        return make_retriever(doc_type="ITS", top_k=8), "ITS only"
    if any(k in q for k in ["article ", "regulation ", "directive ",
                             " shall ", " must ", "2022/2554"]):
        return make_retriever(level=1, top_k=8), "Level 1 (main regulation)"
    return make_retriever(top_k=8), "Full corpus (hybrid)"


def retrieve(query: str, top_k: int = None, verbose: bool = False):
    """Standard retrieval — main entry point used by Phase 5 and Phase 6."""
    retriever, scope = route_query(query)
    results = retriever.retrieve(query)
    if verbose:
        print(f"Scope   : {scope}")
        show_results(results)
    return results, scope


print("✓ Query router ready (make_retriever pattern)")
print("  TLPT route: full corpus — includes Article 26 from L1")


✓ Query router ready (make_retriever pattern)
  TLPT route: full corpus — includes Article 26 from L1


In [29]:
# Cell 14 — HyDE: Hypothetical Document Embedding
#
# TECHNIQUE: Pre-Retrieval — HyDE (Hypothetical Document Embedding)
# Reference: Gao et al. (2022), 'Precise Zero-Shot Dense Retrieval without Relevance Labels'
#
# PROBLEM IT SOLVES:
#   A short user question has a very different embedding profile to a long,
#   information-dense regulatory chunk — even when they are semantically related.
#   For example, 'What are the penalties for misclassifying an incident?' is short
#   and abstract, while Article 50 is long and uses precise legal language.
#   Comparing these vectors is like comparing apples to oranges.
#
# HOW IT WORKS:
#   1. Ask the LLM to generate a HYPOTHETICAL regulatory passage that would
#      answer the question — as if writing actual DORA regulatory text.
#   2. Embed that hypothesis instead of the raw question.
#   3. The hypothesis looks much more like real document chunks, so cosine
#      similarity search becomes significantly more accurate.
#
# PIPELINE CHANGE:
#   Standard:  Question → embed(question) → vector search → chunks
#   HyDE:      Question → LLM → hypothesis → embed(hypothesis) → vector search → chunks
#
# NOTE: HyDE uses one Groq 70b call per query. Routing still uses the original
# question so metadata filters are correctly applied.

HYDE_PROMPT_TEMPLATE = """You are an expert in EU regulatory law, specialising in
DORA (Digital Operational Resilience Act - Regulation (EU) 2022/2554) and its
associated Level 2 regulatory technical standards.

Write a short passage (4-6 sentences) that could plausibly appear in the DORA
regulation or one of its technical standards and that would directly answer the
question below. Write as actual regulatory text - use formal language, refer to
Articles where appropriate, and use SHALL/SHOULD/MAY for obligations.
Do NOT answer the question conversationally. Write as if you are the regulation itself.

Question: {question}

Hypothetical regulatory passage:"""


def llm_complete_with_retry(prompt: str, max_retries: int = 4) -> str:
    """Call Groq 70b with automatic retry on rate limit errors."""
    for attempt in range(1, max_retries + 1):
        try:
            return llm.complete(prompt).text.strip()
        except Exception as e:
            err = str(e)
            is_rate = any(k in err for k in [
                'rate_limit_exceeded', 'RateLimitError', 'tokens per',
                'requests per', '429', 'retry in'
            ])
            if not is_rate:
                raise
            if attempt == max_retries:
                raise RuntimeError(
                    f'Rate limit not resolved after {max_retries} retries.\n'
                    f'Groq daily token limit may be exhausted — try again tomorrow.'
                ) from e
            # Parse retry time from Groq error message
            match = _re.search(r'try again in (\d+(?:\.\d+)?)s', err, _re.IGNORECASE)
            wait  = float(match.group(1)) + 2 if match else 30
            print(f'    Groq rate limit (attempt {attempt}/{max_retries}). Waiting {wait:.0f}s...')
            time.sleep(wait)


def generate_hypothesis(question: str) -> str:
    """
    Generate a hypothetical DORA regulatory passage for a given question.
    This is the core of HyDE — the hypothesis looks like a real document chunk.
    Uses Groq llama-3.3-70b-versatile.
    """
    prompt = HYDE_PROMPT_TEMPLATE.format(question=question)
    return llm_complete_with_retry(prompt)


def retrieve_with_hyde(query: str, verbose: bool = False) -> tuple:
    """
    HyDE retrieval pipeline:
    1. Generate a hypothetical regulatory passage via Groq 70b
    2. Route using the ORIGINAL query (preserves metadata filtering logic)
    3. Retrieve using the HYPOTHESIS embedding (better vector similarity)
    Returns (nodes, scope, hypothesis_text)
    """
    # Step 1: Generate hypothesis
    hypothesis = generate_hypothesis(query)

    # Step 2: Route using original query (correct metadata filters)
    retriever, scope = route_query(query)

    # Step 3: Retrieve using hypothesis as the search query
    nodes = retriever.retrieve(hypothesis)

    if verbose:
        print(f"\nQuery     : {query}")
        print(f"Scope     : {scope}")
        print(f"Hypothesis: {hypothesis[:200]}...")
        print()
        show_results(nodes)

    return nodes, scope, hypothesis


print("✓ HyDE functions ready (Groq llama-3.3-70b-versatile)")
print("  generate_hypothesis(question) — generates hypothetical regulatory passage")
print("  retrieve_with_hyde(query)     — full HyDE retrieval pipeline")


✓ HyDE functions ready (Groq llama-3.3-70b-versatile)
  generate_hypothesis(question) — generates hypothetical regulatory passage
  retrieve_with_hyde(query)     — full HyDE retrieval pipeline


In [30]:
# Cell 15 — HyDE demonstration: standard vs HyDE retrieval
#
# Compares standard retrieval against HyDE on 3 representative queries.
# Shows the generated hypothesis and how it changes which chunks are retrieved.
# Uses 1 Groq 70b call per query (3 calls total).

HYDE_DEMO_QUERIES = [
    # Adversarial — short query, answer buried in penalty articles
    ("What happens if a financial entity fails to report an ICT incident on time?",
     "adversarial"),
    # Vague / general — would benefit from hypothesis expansion
    ("How should financial entities manage their exposure to third-party ICT providers?",
     "general"),
    # Cross-level — needs both L1 and L2 content
    ("What are the detailed criteria for classifying an ICT incident as major?",
     "cross_level"),
]

print("HyDE DEMONSTRATION — Standard vs HyDE Retrieval")
print("=" * 65)
print("(Uses 3 Groq 70b calls — ~10s total)\n")

hyde_improvements = []

for query, qtype in HYDE_DEMO_QUERIES:
    print(f"{'─'*65}")
    print(f"[{qtype.upper()}] {query}")

    # Standard retrieval — embed the raw question
    std_nodes, std_scope = retrieve(query)
    std_srcs = [
        (n.metadata.get('article_num', '?'), n.metadata.get('source_doc', '')[:30])
        for n in std_nodes[:5]
    ]

    # HyDE retrieval — embed a hypothetical regulatory passage instead
    hyde_nodes, hyde_scope, hypothesis = retrieve_with_hyde(query)
    hyde_srcs = [
        (n.metadata.get('article_num', '?'), n.metadata.get('source_doc', '')[:30])
        for n in hyde_nodes[:5]
    ]

    print(f"\nHypothesis (first 250 chars):")
    print(f"  {hypothesis[:250]}...")

    print(f"\nStandard top-5  (embedding: raw question):")
    for art, src in std_srcs:
        print(f"  Art {str(art):<4}  {src}")

    print(f"\nHyDE top-5  (embedding: hypothetical passage):")
    for art, src in hyde_srcs:
        print(f"  Art {str(art):<4}  {src}")

    # Count how many unique chunks HyDE found that standard missed
    std_set  = {n.id_ for n in std_nodes[:5]}
    hyde_set = {n.id_ for n in hyde_nodes[:5]}
    new_chunks = hyde_set - std_set
    print(f"\nNew chunks found by HyDE (not in standard top-5): {len(new_chunks)}")
    hyde_improvements.append(len(new_chunks))

    time.sleep(3)   # polite delay between Groq calls

print(f"\n{'='*65}")
print(f"Average new chunks per query: {sum(hyde_improvements)/len(hyde_improvements):.1f}")
print("Higher = HyDE found more unique results vs standard retrieval")
print("\nConclusion: HyDE improves retrieval by embedding a hypothesis that")
print("looks like the target document rather than the short raw question.")

HyDE DEMONSTRATION — Standard vs HyDE Retrieval
(Uses 3 Groq 70b calls — ~10s total)

─────────────────────────────────────────────────────────────────
[ADVERSARIAL] What happens if a financial entity fails to report an ICT incident on time?

Hypothesis (first 250 chars):
  In the event of a failure to report an ICT incident within the timeframe specified in Article 24(1), the financial entity SHALL submit a reasoned explanation for the delay to the relevant competent authority, as referred to in Article 46, without und...

Standard top-5  (embedding: raw question):
  Art 17    L1_DORA_Regulation_2022_2554
  Art 9     L2_ITS_ICT_Incident_Reporting
  Art 9     L2_ITS_ICT_Incident_Reporting
  Art 37    L1_DORA_Regulation_2022_2554
  Art 19    L1_DORA_Regulation_2022_2554

HyDE top-5  (embedding: hypothetical passage):
  Art 19    L1_DORA_Regulation_2022_2554
  Art 6     L1_DORA_Regulation_2022_2554
  Art 5     L2_RTS_ICT_Incident_Reporting_
  Art 19    L1_DORA_Regulation_2022_2554
  Art 

In [31]:
# Cell 14 — Test query router
routing_tests = [
    "What does the RTS on ICT risk management say about governance?",
    "TLPT frequency and scoping requirements",
    "What are the incident reporting templates?",
    "Guidelines on estimating aggregate costs of ICT incidents",
    "Article 19 requirements for ICT incident classification",
    "Does DORA apply to investment firms?",
]

print("QUERY ROUTER TESTS")
print("=" * 65)
for query in routing_tests:
    _, scope = route_query(query)
    print(f"  Query : {query[:60]}")
    print(f"  Routed: {scope}\n")

QUERY ROUTER TESTS
  Query : What does the RTS on ICT risk management say about governanc
  Routed: Level 2 (RTS/ITS)

  Query : TLPT frequency and scoping requirements
  Routed: Full corpus (TLPT)

  Query : What are the incident reporting templates?
  Routed: ITS only

  Query : Guidelines on estimating aggregate costs of ICT incidents
  Routed: Level 3 (guidelines)

  Query : Article 19 requirements for ICT incident classification
  Routed: Level 1 (main regulation)

  Query : Does DORA apply to investment firms?
  Routed: Full corpus (hybrid)



In [32]:
# Cell 15 — Retrieval evaluation (hit rate)
EVAL_SET = [
    ("Timelines for major ICT incident reporting",                  "L1_DORA_Regulation",              19),
    ("Management body responsibilities for ICT risk",               "L1_DORA_Regulation",               5),
    ("General principles ICT third-party risk management",          "L1_DORA_Regulation",              28),
    ("Contractual provisions ICT third-party service providers",    "L1_DORA_Regulation",              30),
    ("Business continuity and disaster recovery plans",             "L1_DORA_Regulation",              11),
    ("ICT incident classification criteria thresholds",             "L2_RTS_ICT_Incident_Class",        None),
    ("Threat-led penetration testing TLPT scope",                   "L2_RTS_TLPT",                      1),
    ("ICT third-party subcontracting chain requirements",           "L2_RTS_Subcontracting",            None),
    ("Register of information ICT third-party providers",           "L2_ITS_Register_of_Information",   None),
    ("Estimating aggregate annual costs ICT incidents guidelines",  "L3_Guidelines_Costs_and_Losses", None),
]

source_hits = 0
art_hits    = 0
full_hits   = 0
total       = len(EVAL_SET)

print(f"Retrieval evaluation ({total} queries)\n")
print(f"  {'#':<3} {'Src':>4} {'Art':>4}  Query")
print("  " + "─" * 60)

for i, (query, exp_src, exp_art) in enumerate(EVAL_SET, 1):
    results, _ = retrieve(query, top_k=8, verbose=False)
    retrieved_srcs = [n.metadata.get("source_doc", "") for n in results]
    retrieved_arts = [n.metadata.get("article_num", -1) for n in results]

    src_hit = any(exp_src in s for s in retrieved_srcs)
    art_hit = (exp_art is None) or (exp_art in retrieved_arts)

    if src_hit: source_hits += 1
    if art_hit: art_hits    += 1
    if src_hit and art_hit: full_hits += 1

    print(f"  {i:<3} {'✓' if src_hit else '✗':>4} {'✓' if art_hit else '✗':>4}  {query[:50]}")

print("  " + "─" * 60)
print(f"\n  Source hit rate  : {source_hits}/{total} ({100*source_hits//total}%)")
print(f"  Article hit rate : {art_hits}/{total} ({100*art_hits//total}%)")
print(f"  Full hit rate    : {full_hits}/{total} ({100*full_hits//total}%)")
print(f"\n  Baseline target: >70% full hit rate")
status = "✓ READY for Phase 5" if full_hits/total >= 0.7 else "⚠ Consider tuning top_k or chunk size"
print(f"  {status}")

Retrieval evaluation (10 queries)

  #    Src  Art  Query
  ────────────────────────────────────────────────────────────
  1      ✓    ✗  Timelines for major ICT incident reporting
  2      ✓    ✓  Management body responsibilities for ICT risk
  3      ✓    ✓  General principles ICT third-party risk management
  4      ✓    ✓  Contractual provisions ICT third-party service pro
  5      ✓    ✓  Business continuity and disaster recovery plans
  6      ✓    ✓  ICT incident classification criteria thresholds
  7      ✗    ✗  Threat-led penetration testing TLPT scope
  8      ✓    ✓  ICT third-party subcontracting chain requirements
  9      ✓    ✓  Register of information ICT third-party providers
  10     ✓    ✓  Estimating aggregate annual costs ICT incidents gu
  ────────────────────────────────────────────────────────────

  Source hit rate  : 9/10 (90%)
  Article hit rate : 8/10 (80%)
  Full hit rate    : 8/10 (80%)

  Baseline target: >70% full hit rate
  ✓ READY for Phase 5


In [33]:
# Cell 16 — Save retrieval config for Phase 5
retrieval_config = {
    "created_at":           datetime.datetime.now().isoformat(),
    "llm":                  "groq/llama-3.3-70b-versatile",
    "semantic_top_k":       5,
    "bm25_top_k":           5,
    "hybrid_top_k":         5,
    "fusion_mode":          "reciprocal_rerank",
    "query_router":         "keyword_based",
    "retrievers_available": [
        "semantic", "bm25", "hybrid",
        "level1_filtered", "level2_filtered", "level3_filtered",
        "rts_filtered", "its_filtered"
    ],
    "default_retriever":    "hybrid",
    "eval_hit_rate":        f"{full_hits}/{total} ({100*full_hits//total}%)",
    "notes": (
        "Settings.llm must be set to Groq before constructing QueryFusionRetriever "
        "to prevent LlamaIndex from falling back to OpenAI. "
        "Query router uses keyword matching — upgrade to LLM router in Phase 5."
    )
}

retrieval_config_path = BASE_DIR / "retrieval_config.json"
retrieval_config_path.write_text(json.dumps(retrieval_config, indent=2))
print(f"✓ Retrieval config saved: {retrieval_config_path}")
print(json.dumps(retrieval_config, indent=2))

✓ Retrieval config saved: dora_rag/retrieval_config.json
{
  "created_at": "2026-04-15T14:55:50.759810",
  "llm": "groq/llama-3.3-70b-versatile",
  "semantic_top_k": 5,
  "bm25_top_k": 5,
  "hybrid_top_k": 5,
  "fusion_mode": "reciprocal_rerank",
  "query_router": "keyword_based",
  "retrievers_available": [
    "semantic",
    "bm25",
    "hybrid",
    "level1_filtered",
    "level2_filtered",
    "level3_filtered",
    "rts_filtered",
    "its_filtered"
  ],
  "default_retriever": "hybrid",
  "eval_hit_rate": "8/10 (80%)",
  "notes": "Settings.llm must be set to Groq before constructing QueryFusionRetriever to prevent LlamaIndex from falling back to OpenAI. Query router uses keyword matching \u2014 upgrade to LLM router in Phase 5."
}


In [34]:
# Cell 20 — Phase 4 summary
print("\n" + "=" * 65)
print("PHASE 4 COMPLETE — RETRIEVAL PIPELINE + HyDE")
print("=" * 65)
print(f"""
Retrievers implemented:
  Semantic   — BAAI/bge-large-en-v1.5 vector similarity
  BM25       — keyword matching (rank-bm25)
  Hybrid     — RRF fusion of semantic + BM25  ← default
  Filtered   — scoped by level (1/2/3) or doc_type (RTS/ITS)

Advanced techniques:
  Hybrid Search (#4) — BM25 + vector, RRF fusion (fully operational)
  HyDE (#2)          — Hypothetical Document Embedding (pre-retrieval)
                       Gao et al. (2022): embed LLM hypothesis instead
                       of raw query for better vector similarity

LLM setup (Groq only — no other providers needed):
  Settings.llm : Groq llama-3.1-8b-instant   (router internals only)
  llm          : Groq llama-3.3-70b-versatile (HyDE hypothesis generation)

Entry points for Phase 5:
  results, scope        = retrieve(query)              # standard hybrid
  results, scope, hyp   = retrieve_with_hyde(query)    # HyDE

Eval result  : {full_hits}/{total} full hit rate ({100*full_hits//total}%) — standard

Next step -> open 05_generation.ipynb
""")



PHASE 4 COMPLETE — RETRIEVAL PIPELINE + HyDE

Retrievers implemented:
  Semantic   — BAAI/bge-large-en-v1.5 vector similarity
  BM25       — keyword matching (rank-bm25)
  Hybrid     — RRF fusion of semantic + BM25  ← default
  Filtered   — scoped by level (1/2/3) or doc_type (RTS/ITS)

Advanced techniques:
  Hybrid Search (#4) — BM25 + vector, RRF fusion (fully operational)
  HyDE (#2)          — Hypothetical Document Embedding (pre-retrieval)
                       Gao et al. (2022): embed LLM hypothesis instead
                       of raw query for better vector similarity

LLM setup (Groq only — no other providers needed):
  Settings.llm : Groq llama-3.1-8b-instant   (router internals only)
  llm          : Groq llama-3.3-70b-versatile (HyDE hypothesis generation)

Entry points for Phase 5:
  results, scope        = retrieve(query)              # standard hybrid
  results, scope, hyp   = retrieve_with_hyde(query)    # HyDE

Eval result  : 8/10 full hit rate (80%) — standard

Nex